# 02 — LiH Benchmark: CCSD vs VQE vs FCI

LiH (STO-3G) with a 4-electron/4-orbital active space.

## Goals
1. Compare all post-HF methods on accuracy vs cost.
2. Show VQE-UCCSD vs ADAPT-VQE tradeoff (parameters vs accuracy).
3. QPE gives exact ground state with quantum resources only.
4. Energy error bar chart + qubit count table.


In [ ]:
import sys, os, warnings
from pathlib import Path
warnings.filterwarnings('ignore')

pkg_root = Path.cwd().parent
if str(pkg_root) not in sys.path:
    sys.path.insert(0, str(pkg_root))

import quantum_chem_bench.classical_solvers  # noqa
import quantum_chem_bench.quantum_solvers    # noqa

from quantum_chem_bench.core.config import BenchConfig
from quantum_chem_bench.core.runner import BenchRunner
from quantum_chem_bench.analysis.benchmark import BenchmarkPlotter, format_table
import pandas as pd

In [ ]:
config = BenchConfig.from_yaml('../configs/lih_sto3g.yaml')

if os.environ.get('CI_FAST_NB'):
    config.solvers.quantum = ['vqe_uccsd']
    config.solvers.classical = ['hf', 'ccsd', 'fci']

runner = BenchRunner(config, verbose=True)
bench = runner.run()

## Results

In [ ]:
print(format_table(bench, fmt='text'))
print(f"\nFCI energy (exact): {bench.fci_energy:.10f} Ha")

In [ ]:
%matplotlib inline
import matplotlib; matplotlib.rcParams['figure.dpi'] = 120

plotter = BenchmarkPlotter(figsize=(10, 5))
fig = plotter.energy_bar(
    bench, reference='fci',
    title='LiH (STO-3G, 4e/4o active space) — Energy Error vs FCI',
)
fig.savefig('lih_energy_errors.png', dpi=150)
fig

## Quantum resource comparison

In [ ]:
q_data = []
for name, r in bench.results.items():
    if r.n_qubits is not None:
        err = (r.energy - bench.fci_energy) * 1000 if bench.fci_energy else None
        q_data.append({
            'Method': name,
            'N Qubits': r.n_qubits,
            'Energy (Ha)': f'{r.energy:+.8f}',
            'Error vs FCI (mHa)': f'{err:.3f}' if err is not None else '—',
            'Wall Time (s)': f'{r.wall_time:.2f}',
            'Ansatz Params': r.extra.get('ansatz_params', '—'),
        })

pd.DataFrame(q_data)

## Custom active space

Change the active space in one line:

In [ ]:
from quantum_chem_bench.core.interfaces import MolSpec
from quantum_chem_bench.core.registry import registry

# Try larger active space: 2 electrons, 2 orbitals
mol_small = MolSpec(
    geometry='Li 0 0 0; H 0 0 1.595',
    basis='sto-3g',
    n_active_electrons=(1, 1),
    n_active_orbitals=2,
)

fci_s = registry.build('fci', category='solver')
r = fci_s.solve(mol_small)
print(f'LiH 2e/2o FCI: {r.energy:+.10f} Ha  (t={r.wall_time:.2f}s)')